# 5 — The autodiff cheat sheet

Reverse-mode AD for this language, end to end: the region IR, the adjoint of
every operation (with a one-line derivation each), numerical validation, and
the chain-rule machinery that assembles them.

**The one derivation trick.** Every layout op is a *linear* map L on the
value space. Define the inner product ⟨a, b⟩ = Σ over all coordinates of
a·b. Then the cotangent (gradient) pulls back through the *adjoint*:

    ⟨L x, y⟩ = ⟨x, L† y⟩      ⇒      x̄ = L†(ȳ)

Derive any adjoint by writing the left side as a sum and re-grouping it
around x. Nonlinear ops (pointwise markers) contribute their per-element
partial times the incoming cotangent — the chain rule. Two more rules
complete the system: **fan-out accumulates** (a value used twice gets its
cotangents added) and **gradient-free carriers** (bool/int values — masks,
iota, comparisons — carry no cotangent).

In [1]:
import numpy as np
from nbhelp import show  # also puts tensorlib on sys.path
from pdum.dsl.printer import print_program
from pdum.tl import Tensor, red, reduce, scan
from pdum.tl.autodiff import grad, numeric_grad
from pdum.tl.compute import pointwise
from pdum.tl.dialect import run_named, walk_region
from pdum.tl.lifting import lift_step
from pdum.tl.markers import exp, log, neg

## The IR in sixty seconds

A step function — plain Python, one op per assignment — lowers through
`lift_step` into a dialect Region: a straight line of named values. No
branching: the tape IS the region, read backwards.

In [2]:
def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)


def validate(fn, inputs, wrt, target=None):
    ls = lift_step(fn, **inputs)
    target = target or ls.outputs[0]
    rg = grad(ls.region, target, dict(inputs), names=ls.names)
    vals = run_named(rg.region, inputs, rg.names)
    got = vals[rg.grads[wrt]].to_numpy(order=inputs[wrt].names)
    want = numeric_grad(ls.region, target, wrt, inputs, ls.names)
    ok = np.allclose(got, want, rtol=1e-4, atol=1e-6)
    print(("OK " if ok else "FAIL") + f"  d{target}/d{wrt}   max|Δ| = {np.abs(got - want).max():.2e}")
    return ls, rg, vals


rng = np.random.default_rng(0)

## repeat ⊣ reduce-sum — the fundamental pair

    ⟨repeat(x), y⟩ = Σᵢₙ xᵢ yᵢₙ = Σᵢ xᵢ (Σₙ yᵢₙ) = ⟨x, reduceSumₙ(y)⟩

Broadcast's adjoint is summation, and vice versa. Everything with
"normalization" in its name is built from this pair.

In [3]:
def f(x, w):
    r = x.repeat("n", (0, 3))
    m = r * w
    y = reduce(red.sum, m, ("i", "n"))
    return y


validate(f, {"x": T(rng.standard_normal(4), ("i",)), "w": T(rng.standard_normal((4, 3)), ("i", "n"))}, "x");

OK   dy/dx   max|Δ| = 1.60e-10


## slice ⊣ pad — restriction and zero-extension

    ⟨slice(x), y⟩ = Σ_{i∈S} xᵢ yᵢ = ⟨x, pad₀(y)⟩

The cotangent of a slice is the cotangent zero-extended back to the full
domain — and pad's adjoint is the slice, with cotangent arriving in the
fill region *discarded* (fill is a constant; nothing flows to it).

In [4]:
def f(x):
    s = x.slice(i=(1, 4))
    y = reduce(red.sum, s, "i")
    return y


ls, rg, vals = validate(f, {"x": T([1.0, 2, 3, 4, 5], ("i",))}, "x")
show(vals[rg.grads["x"]], "d y / d x  — ones inside the slice, zeros outside")
print(vals[rg.grads["x"]].to_numpy())

OK   dy/dx   max|Δ| = 1.03e-09
-- d y / d x  — ones inside the slice, zeros outside
Tensor[float64] on Buffer(8B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  i            0      0      5      5  
  numel=5  footprint=(0, 8)  injectivity=unknown
  Guard(1 <= i < 4)
  fill   : 0.0
[0. 1. 1. 1. 0.]


## Relabelings are (almost) their own adjoints

shift ⊣ shift-back, flip ⊣ flip, rename ⊣ inverse-rename, split ⊣ merge,
merge ⊣ split — coordinate relabelings move the cotangent by the inverse
relabeling. (The generated split-adjoint inserts a `materialize` before its
merge: merging needs real stride nesting, and a cotangent arriving from an
arbitrary chain may not have it.)

In [5]:
def f(x, w):
    b = x.split("i", io=2, ii=3)
    m = b * w
    g = m.merge(("io", "ii"), "i")
    y = reduce(red.sum, g, "i")
    return y


validate(f, {"x": T(rng.standard_normal(6), ("i",)), "w": T(rng.standard_normal((2, 3)), ("io", "ii"))}, "x");

OK   dy/dx   max|Δ| = 9.49e-11


## window / stencil ⊣ overlap-add — the convolution adjoint

    ⟨W x, y⟩ = Σₐₖ x_{a+k} yₐₖ = Σⱼ xⱼ (Σₖ y_{j−k,k})   ⇒   x̄ⱼ = Σₖ ȳ_{j−k,k}

Overlapping windows read each xⱼ many times; the adjoint *adds back* each
tap's cotangent, shifted. The transform unrolls per tap: select the tap,
shift by it, slice to the valid region (a stencil's out-of-guard taps die
here — fill gets no gradient), pad with zeros, accumulate.

In [6]:
def f(x, w):
    xs = x.stencil("i", (-1, 1), None, 0.0)
    wr = w.repeat("i", (0, 5))
    m = xs * wr
    y = reduce(red.sum, m, ("i", "i_k"))
    return y


inputs = {"x": T(rng.standard_normal(5), ("i",)), "w": T(rng.standard_normal(3), ("i_k",)).shift(i_k=-1)}
ls, rg, vals = validate(f, inputs, "x")
print()
print("the generated overlap-add (backward excerpt):")


def attrs(n):  # thaw the frozen dict-valued params for display
    return {k: dict(v) if isinstance(v, tuple) and v and isinstance(v[0], tuple) else v for k, v in n.attrs}


fwd_ids = {id(n) for n in walk_region(ls.region)}
for n in walk_region(rg.region):
    if id(n) not in fwd_ids and n.op in ("tl.select", "tl.shift", "tl.slice", "tl.pad"):
        print(f"  {n.op[3:]:<7} {attrs(n)}")

OK   dy/dx   max|Δ| = 7.97e-11

the generated overlap-add (backward excerpt):
  select  {'coords': {'i_k': -1}}
  shift   {'deltas': {'i': -1}}
  slice   {'ranges': {'i': (0, 4)}}
  pad     {'extents': {'i': (0, 5)}, 'fill': 0.0}
  select  {'coords': {'i_k': 0}}
  shift   {'deltas': {'i': 0}}
  slice   {'ranges': {'i': (0, 5)}}
  pad     {'extents': {'i': (0, 5)}, 'fill': 0.0}
  select  {'coords': {'i_k': 1}}
  shift   {'deltas': {'i': 1}}
  slice   {'ranges': {'i': (1, 5)}}
  pad     {'extents': {'i': (0, 5)}, 'fill': 0.0}


## decimate ⊣ zero-stuffing

    ⟨D x, y⟩ = Σⱼ x_{fj+p} yⱼ   ⇒   x̄ᵢ = ȳ_{(i−p)/f} when i ≡ p (mod f), else 0

The adjoint spreads the cotangent into every f-th slot. Generated as:
repeat a phase dim, mask the kept slot with an iota comparison (masks are
gradient-free by carrier), materialize in interleave order, merge.

In [7]:
def f(x, w):
    d = x.decimate("i", 2, 1)
    m = d * w
    y = reduce(red.sum, m, "i")
    return y


inputs = {"x": T(rng.standard_normal(6), ("i",)), "w": T(rng.standard_normal(3), ("i",))}
ls, rg, vals = validate(f, inputs, "x")
print("gradient:", vals[rg.grads["x"]].to_numpy(), " (zeros on the dropped phase)")

OK   dy/dx   max|Δ| = 1.23e-10
gradient: [0.         1.34587542 0.         0.7813114  0.         0.26445563]  (zeros on the dropped phase)


## diagonal ⊣ masked embedding

    ⟨diag(x), y⟩ = Σ_z x_{z,z} y_z   ⇒   x̄ᵢⱼ = [i = j] · ȳᵢ

The cotangent lands on the diagonal of a zero matrix — built from an iota
equality mask and a zero constant, then padded to the full box.

In [8]:
def f(x):
    d = x.diagonal(("i", "j"), "z")
    y = reduce(red.sum, d, "z")
    return y


ls, rg, vals = validate(f, {"x": T(rng.standard_normal((3, 3)), ("i", "j"))}, "x")
print(vals[rg.grads["x"]].to_numpy(order=("i", "j")))

OK   dy/dx   max|Δ| = 1.40e-10
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## The pointwise marker table

| marker | x̄ (first operand) | second operand |
|---|---|---|
| add | c | c |
| sub | c | −c |
| mul | c·B | c·A |
| div | c/B | −c·A/B² |
| exp | c·out (reuses the forward output!) | — |
| log | c/A | — |
| maximum | c·[A≥B] | c·[B>A] (ties to the first) |
| where(m,a,b) | routed by m | routed by ¬m; m itself: none (bool) |
| comparisons, iota | gradient-free (int/bool carriers) | |

## reduce and scan

| op | adjoint |
|---|---|
| reduce(sum) | repeat |
| reduce(mean) | repeat, then ÷N (N static, from the layout) |
| reduce(max) | repeat, masked by eq(A, max) — tie caveat |
| scan(sum) | reverse scan: flip ∘ scan(sum) ∘ flip |

Derivation for scan: yₜ = Σ_{s≤t} xₛ ⇒ x̄ₛ = Σ_{t≥s} ȳₜ — suffix sums.

In [9]:
def f(x, w):
    cs = scan(red.sum, x, "i")
    m = cs * w
    y = reduce(red.sum, m, "i")
    return y


validate(f, {"x": T(rng.standard_normal(5), ("i",)), "w": T(rng.standard_normal(5), ("i",))}, "x");

OK   dy/dx   max|Δ| = 1.07e-10


## The chain rule, mechanically

`grad` walks the region backwards. Each value collects cotangent
*contributions* from every operation that consumed it; fan-out means
several contributions, summed. Watch x used twice in x·x:

In [10]:
def f(x):
    sq = x * x
    y = reduce(red.sum, sq, "i")
    return y


ls, rg, vals = validate(f, {"x": T([1.0, -2.0, 3.0], ("i",))}, "x")
print("d/dx Σ x² =", vals[rg.grads["x"]].to_numpy(), " (= 2x: two contributions, added)")
print()
print("the full joint region:")
print(print_program(rg.region, "joint"))

OK   dy/dx   max|Δ| = 8.39e-10
d/dx Σ x² = [ 2. -4.  6.]  (= 2x: two contributions, added)

the full joint region:
joint(%p0: TensorType(dims=(i[0:3),))) {
  %0 = tl.pointwise %p0, %p0 {f = 'mul'} : TensorType(dims=(i[0:3),))
  %1 = tl.reduce %0 {dims = 'i', f = 'sum'} : TensorType(dims=())
  %2 = tl.const {dims = (), value = 1.0} : TensorType(dims=())
  %3 = tl.repeat %2 {chart = None, extent = (0, 3), labels = None, name = 'i'} : TensorType(dims=(i[0:3),))
  %4 = tl.with_charts %3 {charts = (('i', None),)} : TensorType(dims=(i[0:3),))
  %5 = tl.pointwise %p0, %p0 {f = 'mul.d0'} : TensorType(dims=(i[0:3),))
  %6 = tl.pointwise %4, %5 {f = 'mul'} : TensorType(dims=(i[0:3),))
  %7 = tl.with_charts %6 {charts = (('i', None),)} : TensorType(dims=(i[0:3),))
  %8 = tl.pointwise %p0, %p0 {f = 'mul.d1'} : TensorType(dims=(i[0:3),))
  %9 = tl.pointwise %4, %8 {f = 'mul'} : TensorType(dims=(i[0:3),))
  %10 = tl.with_charts %9 {charts = (('i', None),)} : TensorType(dims=(i[0:3),))
  %11 = tl.poi

## Seeds: gradients are vector-Jacobian products

A scalar target seeds with 1 automatically. A non-scalar target has a
Jacobian, and reverse mode natively computes v↦Jᵀv — so a seed is
*required*, as an extra runtime input aligned with the target (silent
ones-seeding is a footgun this transform refuses).

In [11]:
def f(x):
    d = x * x
    return d


ls = lift_step(f, x=T([1.0, 2.0], ("i",)))
try:
    grad(ls.region, "d", names=ls.names)
except ValueError as e:
    print("refused:", e)
rg = grad(ls.region, "d", seed="dY", names=ls.names)
vals = run_named(rg.region, {"x": T([3.0, 4.0], ("i",)), "dY": T([1.0, 0.0], ("i",))}, rg.names)
print("row of the Jacobian via seed [1,0]:", vals[rg.grads["x"]].to_numpy())

refused: target 'd' is not a scalar; pass seed= (the name of a runtime input aligned with the target) — reverse mode computes vector-Jacobian products
row of the Jacobian via seed [1,0]: [6. 0.]


## End to end: training with the language

Softmax cross-entropy has the famous analytic gradient softmax(S) − onehot —
the generated region reproduces it. Then: a few steps of gradient descent
on a linear model, driven entirely by generated regions.

In [12]:
s = rng.standard_normal((2, 4))
onehot = np.zeros((2, 4))
onehot[0, 1] = onehot[1, 3] = 1.0


def f(S, t):
    mx = reduce(red.max, S, "v")
    mr = mx.repeat("v", (0, 4))
    sh = S - mr
    e = pointwise(exp, sh)
    z = reduce(red.sum, e, "v")
    zr = z.repeat("v", (0, 4))
    lz = pointwise(log, zr)
    lp = sh - lz
    nll = lp * t
    s1 = reduce(red.sum, nll, "v")
    nl = pointwise(neg, s1)
    L = reduce(red.sum, nl, "i")
    return L


inputs = {"S": T(s, ("i", "v")), "t": T(onehot, ("i", "v"))}
ls = lift_step(f, **inputs)
rg = grad(ls.region, "L", dict(inputs), names=ls.names)
vals = run_named(rg.region, inputs, rg.names)
sm = np.exp(s - s.max(1, keepdims=True))
sm /= sm.sum(1, keepdims=True)
print("max |dL/dS − (softmax − onehot)| =", np.abs(vals[rg.grads["S"]].to_numpy(order=("i", "v")) - (sm - onehot)).max())

max |dL/dS − (softmax − onehot)| = 5.551115123125783e-17


In [13]:
xd = rng.standard_normal((8, 3))
true_w = np.array([2.0, -1.0, 0.5])
yd = xd @ true_w


def model(X, Wp, Y):
    X3 = X.repeat("o", (0, 1))
    W3 = Wp.repeat("i", (0, 8))
    P = X3 * W3
    yh = reduce(red.sum, P, "k")
    Y3 = Y.repeat("o", (0, 1))
    r = yh - Y3
    r2 = r * r
    L = reduce(red.mean, r2, ("i", "o"))
    return L


w = np.zeros(3)
ls = lift_step(model, X=T(xd, ("i", "k")), Wp=T(np.zeros((3, 1)), ("k", "o")), Y=T(yd, ("i",)))
rg = grad(ls.region, "L", {"X": T(xd, ("i", "k")), "Wp": T(np.zeros((3, 1)), ("k", "o")), "Y": T(yd, ("i",))}, names=ls.names)
for step in range(60):
    inputs = {"X": T(xd, ("i", "k")), "Wp": T(w[:, None], ("k", "o")), "Y": T(yd, ("i",))}
    vals = run_named(rg.region, inputs, rg.names)
    g = vals[rg.grads["Wp"]].to_numpy(order=("k", "o"))[:, 0]
    w -= 0.4 * g
    if step % 20 == 0 or step == 59:
        print(f"step {step:2d}   loss = {float(vals['L'].item()):.6f}")
print("learned w =", np.round(w, 4), "   true w =", true_w)

step  0   loss = 6.637535
step 20   loss = 0.000000
step 40   loss = 0.000000
step 59   loss = 0.000000
learned w = [ 2.  -1.   0.5]    true w = [ 2.  -1.   0.5]


---
Notes and known gaps (CONCERNS #19–21): gradients carry their primal's
coordinate charts and labels (the entry for the sample at 0.75 um is
labeled 0.75 um), and with `target_unit=` their value units transform as
unit(L)/unit(v); decimate's adjoint needs a factor-divisible domain; only
scan(sum) and reduce(sum/mean/max/min) differentiate so far; n-ary diagonal
adjoints and pair-state scans await the marker DSL. Next layer up:
REPRESENTATIONS.md — schedules, checkpointing, and memory.